In [ ]:
# Bước 1: Thiết lập biến môi trường offline, fix random seed và import module suy luận
import json
import os
import sys
import time
from pathlib import Path
from tqdm import tqdm

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
os.environ.setdefault("VLLM_NO_USAGE_STATS", "1")
os.environ.setdefault("DO_NOT_TRACK", "1")
os.environ.setdefault("PYTHONUTF8", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

CODE_ROOT = Path("/code") if Path("/code/agent.py").exists() else Path.cwd()
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from agent import (
    DEFAULT_CONFIG_PATH,
    build_audio_capped_chunks,
    discover_split,
    import_zaic_tools,
    infer_batch,
    infer_sample,
    load_models,
    seed_everything,
)
from src.core.dataset import load_catalog, load_split

seed_everything(42)
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/data"))
OUTPUT_PATH = Path(os.environ.get("OUTPUT_PATH", "/result/jupyter_submission.jsonl"))
CONFIG_PATH = Path(os.environ.get("CONFIG_PATH", str(DEFAULT_CONFIG_PATH)))
print(f"CODE_ROOT={CODE_ROOT} | DATA_ROOT={DATA_ROOT} | OUTPUT_PATH={OUTPUT_PATH}")

In [ ]:
# Bước 2: Load offline models (Qwen3.5-4B SFT V2 + PhoWhisper-Large-ZaloAI)
# Hỗ trợ inject model giả lập qua MODULE hook khi chạy unit test; mặc định load model offline thật.
t_load_start = time.perf_counter()
_inj_asr = globals().get("_INJECTED_ASR_ENGINE")
_inj_llm = globals().get("_INJECTED_LLM_BACKEND")
cfg, raw_cfg, asr_engine, llm_backend = load_models(
    CONFIG_PATH,
    asr_engine=_inj_asr,
    llm_backend=_inj_llm,
)
load_ms = (time.perf_counter() - t_load_start) * 1000.0
print(f"load model: {load_ms:.2f} ms")

In [ ]:
# Bước 3: Khám phá động split trong /data, import zaic_tools từ split và nạp catalog + samples
SPLIT = discover_split(DATA_ROOT, split_override=os.environ.get("DATA_SPLIT") or None)
zaic_tools = import_zaic_tools(DATA_ROOT, SPLIT)
catalog = load_catalog(SPLIT, DATA_ROOT)
test_cases = load_split(SPLIT, DATA_ROOT)

_limit_env = os.environ.get("SAMPLE_LIMIT")
if _limit_env:
    test_cases = test_cases[: int(_limit_env)]

print(f"SPLIT={SPLIT} | total_samples={len(test_cases)} | zaic_tools={zaic_tools.__file__}")

In [ ]:
# Bước 4: Chạy suy luận tối ưu batch theo audio budget (giữ nguyên thứ tự), đo time từ trước bước đọc audio, ghi /result/jupyter_submission.jsonl
seed_everything(cfg.seed)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
_strict_mode = os.environ.get("STRICT_INFERENCE", "0") == "1"
_batch_size = max(1, int(getattr(cfg.agent, "batch_size", 1)))

results = []
t_pred_start = time.perf_counter()
with open(OUTPUT_PATH, "w", encoding="utf-8", newline="\n") as fout, tqdm(total=len(test_cases), desc=f"Infer {SPLIT}") as pbar:
    if _batch_size == 1:
        for item in test_cases:
            row = infer_sample(
                item,
                split=SPLIT,
                data_root=DATA_ROOT,
                catalog=catalog,
                asr_engine=asr_engine,
                llm_backend=llm_backend,
                cfg=cfg,
                raw_cfg=raw_cfg,
                strict=_strict_mode,
            )
            fout.write(json.dumps(row, ensure_ascii=False) + "\n")
            fout.flush()
            results.append(row)
            pbar.update(1)
    else:
        for chunk in build_audio_capped_chunks(test_cases, max_batch_size=_batch_size, max_chunk_audios=8):
            chunk_rows = infer_batch(
                chunk,
                split=SPLIT,
                data_root=DATA_ROOT,
                catalog=catalog,
                asr_engine=asr_engine,
                llm_backend=llm_backend,
                cfg=cfg,
                raw_cfg=raw_cfg,
                strict=_strict_mode,
            )
            for row in chunk_rows:
                fout.write(json.dumps(row, ensure_ascii=False) + "\n")
            fout.flush()
            results.extend(chunk_rows)
            pbar.update(len(chunk_rows))

pred_ms = (time.perf_counter() - t_pred_start) * 1000.0
avg_ms = pred_ms / max(1, len(results))
print(f"predict: {len(results)} mẫu, {pred_ms:.2f} ms, trung bình {avg_ms:.2f} ms/mẫu")
print(f"Saved -> {OUTPUT_PATH}")